In [ ]:
# !pip install minio delta-spark

In [ ]:
import os
from myutils import remove_location
from minio import Minio
from pyspark.sql import SparkSession
from delta.tables import *
from pyspark.sql import functions as F

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
BUCKET_BRONZE = "bronze"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppAula01") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/tmp/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/tmp/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [ ]:
spark

## Writing using Spark in Traditional Data Lake

In [ ]:
# Example of writing to a MinIO bucket
df = spark.createDataFrame([(1, "apple"), (2, "banana"), (3, "laranja")], ["id", "fruit"])
df.write.format("parquet").mode("overwrite").save("s3a://bronze/test/data.parquet")

In [ ]:
# Reading back
df_loaded = spark.read.format("parquet").load("s3a://bronze/test/data.parquet")
df_loaded.show()

In [ ]:
df_loaded.createOrReplaceTempView("tradicional_dl")

In [ ]:
# spark.sql("UPDATE tradicional_dl SET fruit = 'maça' WHERE id = 3")

In [ ]:
# spark.sql("DELETE FROM tradicional_dl WHERE id = 1")

## Writing our first Delta Table!

### DELETE

In [ ]:
location = "s3a://bronze/test/simple_delete"

In [ ]:
# Example of writing to a MinIO bucket
df_del = spark.createDataFrame([(1, "apple"), (2, "banana"), (3, "laranja")], ["id", "fruit"])
df_del.write.format("delta").mode("overwrite").save(location)

In [ ]:
df_del = spark.read.format('delta').load(location)

In [ ]:
df_del.toPandas().head()

In [ ]:
df_del.createOrReplaceTempView("simple_test_delete")

In [ ]:
spark.sql("DELETE FROM simple_test_delete WHERE id = 2")

In [ ]:
# Create a DeltaTable object
delta_table = DeltaTable.forPath(spark, location)
# Get history of table
history_df = delta_table.history()
history_df.toPandas().head()

In [ ]:
delta_table.history().select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
spark.read.format('delta').option("versionAsOf", "0").load(location).toPandas().head()

In [ ]:
spark.read.format('delta').load(location).toPandas().head()

### UPDATE

In [ ]:
location = "s3a://bronze/test/simple_update"

In [ ]:
df_update = spark.createDataFrame([(1, "apple"), (2, "banana"), (3, "laranja"), (4, "mangua")], ["id", "fruit"])
df_update.write.format("delta").mode("overwrite").save(location)

In [ ]:
df_update = spark.read.format('delta').load(location)
df_update.toPandas().head()

In [ ]:
df_update.createOrReplaceTempView("simple_test_update")

In [ ]:
spark.sql("UPDATE simple_test_update SET fruit = 'mango' WHERE id = 4")

In [ ]:
# Create a DeltaTable object
delta_table = DeltaTable.forPath(spark, location)
# Get history of table
history_df = delta_table.history()
history_df.toPandas().head()

In [ ]:
delta_table.history().select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
spark.read.format('delta').option("versionAsOf", "0").load(location).toPandas().head()

In [ ]:
spark.read.format('delta').load(location).toPandas().head()

### MERGE

Operation using UPSERT with Delta Lake Table
- [documentation](https://docs.delta.io/latest/delta-update.html)

In [ ]:
location = "s3a://bronze/test/simple_merge"

In [ ]:
df_merge = spark.createDataFrame([(1, "apple"), (2, "banana"), (3, "laranja"), (4, "mango")], ["id", "fruit"])
df_merge.write.format("delta").mode("overwrite").save(location)

In [ ]:
delta_table = DeltaTable.forPath(spark, location)

Now let's add a new fruit into Dataframe: `pineapple` at updating from `banana` to `strawberry`

'MERGE` is one of the most powerful #DeltaLake operations. 💪

With Delta Lake merge you can apply multiple INSERT, UPDATE and/or DELETE operations in a single transaction. You can also specify conditional statements for your upserts.

In [ ]:
df_new_data = spark.createDataFrame([(1, "apple"), (2, "strawberry"), (3, "laranja"), (4, "mango"), (5, "pineapple")], ["id", "fruit"])

In [ ]:
delta_table.alias('tb_fruit') \
  .merge(
    df_new_data.alias('updates'),
    'tb_fruit.id = updates.id'
  ) \
  .whenMatchedUpdate(set =
    {
      "id": "updates.id",
      "fruit": "updates.fruit",
    }
  ) \
  .whenNotMatchedInsert(values =
    {
      "id": "updates.id",
      "fruit": "updates.fruit",
    }
  ) \
  .execute()

In [ ]:
spark.read.format('delta').load(location).show()

In [ ]:
# Get history of table
history_df = delta_table.history()
history_df.select(["version", "operation", "operationMetrics"]).show(truncate=False)

In [ ]:
spark.stop()